# PVC Video: illustrations for "पचास गज" (Kaggle version)

Makes every illustration in `colab/shots.json` with **SDXL** (CreativeML Open RAIL++-M: commercial use allowed), his face from the 2003 India Post stamp through **IP-Adapter** (Apache 2.0), and a depth map per image with **Depth Anything V2 Small** (Apache 2.0).

Needs: Accelerator **GPU** and Internet **On** (right-hand panel, Session options). Run it with **Save Version → Save & Run All (Commit)** so it finishes in the background. The result is `frames.zip` in the version's **Output** tab.


In [ ]:
import torch, sys
if not torch.cuda.is_available():
    raise SystemExit('NO GPU. Do this: in the right-hand panel set Accelerator to GPU T4 x2 (or P100) and Internet to On, then run again.')
print('GPU:', torch.cuda.get_device_name(0))
!pip -q install -U "diffusers>=0.31,<0.36" "transformers>=4.46,<5" accelerate safetensors
!rm -rf PVC-Video && git clone -q --depth 1 https://github.com/Eeshansingh/PVC-Video
import os
OUT = '/kaggle/working/frames'
os.makedirs(OUT, exist_ok=True)
print('saving to', OUT)

In [ ]:
import json, torch
from PIL import Image
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
from transformers import CLIPVisionModelWithProjection

cfg = json.load(open('PVC-Video/colab/shots.json'))
enc = CLIPVisionModelWithProjection.from_pretrained('h94/IP-Adapter', subfolder='models/image_encoder', torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0', image_encoder=enc,
                                                 torch_dtype=torch.float16, variant='fp16', use_safetensors=True).to('cuda')
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.load_ip_adapter('h94/IP-Adapter', subfolder='sdxl_models', weight_name='ip-adapter-plus-face_sdxl_vit-h.safetensors')
stamp = Image.open('PVC-Video/film/media/stamp.jpg').convert('RGB')
face = stamp.crop((140, 120, 380, 400)).resize((224, 224))   # the face on the stamp
print('ready')

In [ ]:
import time
def make(shot, w, h, seed, path):
    if os.path.exists(path):
        return
    prompt = shot['prompt'].replace('HERO', cfg['hero']) + ', ' + cfg['style']
    pipe.set_ip_adapter_scale(0.6 if shot.get('face') else 0.0)
    g = torch.Generator('cuda').manual_seed(seed)
    img = pipe(prompt=prompt, negative_prompt=cfg['negative'], ip_adapter_image=face, width=w, height=h,
               num_inference_steps=26, guidance_scale=6.0, generator=g).images[0]
    img.save(path, quality=95)

t0 = time.time()
for i, s in enumerate(cfg['shots']):
    for k in range(2):                                    # two takes of every 16:9 shot
        make(s, 1344, 768, 1000 + i * 10 + k, f"{OUT}/{s['id']}_{k}.jpg")
    if s.get('vertical'):
        make(s, 768, 1344, 5000 + i, f"{OUT}/{s['id']}_v.jpg")
    print(f"{i + 1}/{len(cfg['shots'])} {s['id']}  {int(time.time() - t0)} s", flush=True)
print('all images done')

In [ ]:
# depth maps for parallax (white = near)
del pipe; torch.cuda.empty_cache()
from transformers import pipeline as hf_pipeline
import numpy as np
depth = hf_pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Small-hf', device=0)
for f in sorted(os.listdir(OUT)):
    if not f.endswith('.jpg'):
        continue
    d = f[:-4] + '_depth.png'
    if os.path.exists(f'{OUT}/{d}'):
        continue
    out = np.array(depth(Image.open(f'{OUT}/{f}'))['depth'])
    Image.fromarray(out).save(f'{OUT}/{d}')
print('depth done')

In [ ]:
# one zip to download
import shutil
shutil.make_archive('/kaggle/working/frames', 'zip', OUT)
shutil.rmtree(OUT)   # keep only the zip in the output
print('done: frames.zip is in the Output tab')